

## Google Colab 실습: LangChain Agent & Tool 호출

### [Cell 1] 필수 라이브러리 설치

가장 먼저 LangChain과 OpenAI 연동을 위한 패키지를 설치합니다.

In [ ]:
# LangChain 및 OpenAI 패키지 설치
!pip -q install -U "langchain[openai]"

In [5]:
# ChatOpenAI 초기화에 필요한 langchain-openai 패키지 설치
!pip -q install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 17.9 MB/s eta 0:00:00


### [Cell 2] API 키 설정

OpenAI API를 사용하기 위해 보안상 안전한 방식으로 키를 입력받습니다.

In [1]:
import os
from google.colab import userdata

# Colab Secrets에서 OPENAI_API_KEY를 가져옵니다.
# Colab 노트북의 왼쪽 패널에서 '자물쇠' 아이콘을 클릭하여 'OPENAI_API_KEY' 시크릿을 추가할 수 있습니다.
# 만약 OPENAI_API_KEY가 Colab Secrets에 설정되어 있지 않으면, 아래의 getpass 방식으로 입력하게 됩니다.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
        print("OPENAI_API_KEY를 Colab Secrets에서 성공적으로 불러왔습니다.")
    except Exception as e:
        print(f"Colab Secrets에서 OPENAI_API_KEY를 불러오는 데 실패했습니다: {e}")
        print("Colab Secrets에 'OPENAI_API_KEY'를 설정하거나 다음 셀에서 수동으로 입력해주세요.")

OPENAI_API_KEY를 Colab Secrets에서 성공적으로 불러왔습니다.


In [2]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

### [Cell 3] 라이브러리 임포트 및 Tool 정의

에이전트가 사용할 도구를 정의합니다. `@tool` 데코레이터를 사용하여 함수를 정의하면 LangChain이 자동으로 도구 명세를 생성합니다.

In [3]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

@tool
def add_numbers(a: int, b: int) -> str:
    """두 정수를 더한 결과를 문자열로 반환합니다."""
    result = a + b
    # 도구가 실제로 호출되는지 확인하기 위한 로그
    print(f"\n[TOOL RUN] add_numbers(a={a}, b={b}) -> {result}")
    return str(result)

# Tool 단독 테스트
print("=== Tool 직접 실행 테스트 ===")
tool_result = add_numbers.invoke({"a": 3, "b": 4})
print("Tool result:", tool_result)

=== Tool 직접 실행 테스트 ===

[TOOL RUN] add_numbers(a=3, b=4) -> 7
Tool result: 7


### [Cell 4] 모델 초기화 및 에이전트 생성

최신 `init_chat_model` 패턴을 사용하여 LLM을 초기화하고, 도구를 사용하는 에이전트를 구성합니다.

In [6]:
# 1. 모델 초기화 (gpt-4o 또는 gpt-4-turbo 권장)
model = init_chat_model("gpt-4o", temperature=0)

# 2. 에이전트 생성
# system_prompt를 통해 에이전트의 역할과 도구 사용 지침을 명시합니다.
agent = create_agent(
    model=model,
    tools=[add_numbers],
    system_prompt=(
        "You are a careful math assistant. "
        "When the user asks for an addition, you must use the add_numbers tool first. "
        "After using the tool, answer in Korean briefly."
    ),
)

### [Cell 5] 에이전트 실행 및 추적(Trace) 확인

사용자의 질문을 에이전트에게 전달하고, 에이전트가 사고하는 과정(메시지 흐름)을 출력합니다.

In [7]:
# 에이전트 실행
user_question = "27과 35를 더해 주세요."

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_question}
        ]
    }
)

# 메시지 단계별 추적 출력
print("\n=== Agent message trace ===")
for i, msg in enumerate(result["messages"]):
    print(f"\n[{i}] {type(msg).__name__}")
    print("content:", getattr(msg, "content", ""))

    # 도구 호출 정보가 있는지 확인
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        print("tool_calls:", tool_calls)

# 최종 답변 출력
print("\n" + "="*30)
print("=== 최종 답변 ===")
print(result["messages"][-1].content)


[TOOL RUN] add_numbers(a=27, b=35) -> 62

=== Agent message trace ===

[0] HumanMessage
content: 27과 35를 더해 주세요.

[1] AIMessage
content: 
tool_calls: [{'name': 'add_numbers', 'args': {'a': 27, 'b': 35}, 'id': 'call_vt4k6h4fpfhdvjQg93Fbls9L', 'type': 'tool_call'}]

[2] ToolMessage
content: 62

[3] AIMessage
content: 62입니다.

=== 최종 답변 ===
62입니다.


---

💡 코드 핵심 포인트 요약

1.
**`@tool` 데코레이터**: 복잡한 설정 없이 파이썬 함수에 데코레이터와 Docstring만 추가하면 에이전트가 이해할 수 있는 도구로 변환됩니다.


2. **`init_chat_model`**: 모델 이름을 문자열로 전달하여 유연하게 모델을 교체할 수 있는 최신 인터페이스입니다.
3.
**메시지 추적**: 에이전트 실행 결과의 `messages` 리스트를 확인하면 **사용자 질문 → 모델의 도구 호출 결정 → 도구 실행 결과 → 최종 답변**으로 이어지는 흐름을 한눈에 볼 수 있습니다.


4. **한국어 답변 유도**: `system_prompt` 설정을 통해 도구 사용은 정확하게, 답변은 한국어로 하도록 제어합니다.

이 과정을 성공적으로 마치셨다면, 이후에는 `add_numbers` 외에도 데이터베이스 조회나 API 호출 같은 더 복잡한 기능을 가진 도구를 추가하여 에이전트의 능력을 확장할 수 있습니다.